## LDA topic modeling


The utility of topic modeling methods is their capability to uncover unobserved variables—topics—which shape the meaning of textual documents. Modern-day scholars utilize topic modeling to uncover latent topics from a wide array of textual information—from shorter texts, such as twitter posts to longer texts, such as journal articles.



This notebook applies LDA modeling to an experimental dataset investigating participants' goal inferences. 


### Key python libraries:
- gensim (https://radimrehurek.com/gensim/)
- nltk (https://www.nltk.org)
- spacy (https://spacy.io)

### Helpful Links:
- https://medium.com/@lettier/how-does-lda-work-ill-explain-using-emoji-108abf40fa7d
- https://towardsdatascience.com/light-on-math-machine-learning-intuitive-guide-to-latent-dirichlet-allocation-437c81220158






## A Comprehensive Example:

The data represent textual participants’ responses of 812 participants of the questionnaire described in the paper "A Theory-Driven Computational Measure of the Goal Construct in Communication Science". Responses from a single participant are aggregated together in a single document, leaving a total of 812 documents combined in a single data set.

LDA assumes that a single document can contribute to multiple topics simultaneously; in other words, LDA explicitly models the actual distribution of words within each document. However, this assumption has drawbacks when modeling relatively shorter texts (e.g., twitter posts) because these documents may not contain enough meaningful words to model their distribution across topics. As a result, LDA models of longer texts produce more variance than with shorter texts—raising concerns when modeling open-ended responses.

The goal of the analysis is to investigate participants’ responses of the questionnaire.

##  Steps of the analysis:

#### 1. Preparing data for LDA
    a. Spell check
    b. Expand contractions
    c. Read the data 
    d. Check data integrity
    e. Delete missing values
#### 2. Text preprocessing
    a. Tokenization
    b. Lemmatization  
    c. Stop Word Removal
    d. Bigrams and Trigrams
    e. Exclude terms in > 99% and < 1% of documents
    f. Generate Corpus and Dictionary
#### 3. Model selection (Selecting the number of topics (k))
    a. Computing Model Perplexity
    b. Analyzing model results through pyLDAvis visualization
    c. Saving selected model results

In [1]:
## Load Required Libraries

#general
import numpy as np
import pandas as pd
import re
import pickle
from IPython.display import display

#setting up Jupyter notebook 
%matplotlib inline
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', 5000)
pd.set_option('display.width', 10000)

#text preprocessing
import nltk
from nltk.corpus import stopwords

import spacy
from spacy.lang.en import English

from gensim.models import Phrases
from gensim.utils import simple_preprocess


#modeling
import gensim
from gensim.models.ldamodel import LdaModel


#plotting
import pyLDAvis
import pyLDAvis.gensim

### 1. Preparing data for LDA

Before you read in your data, you should manually run your textual data through a spell checker in order to take advantage of the semantic and syntactic context when selecting the proper correct spelling.

Similar to the spellchecker, we needed human coders to expand all English contractions (e.g., "don't" -> "do not"), to ensure accuracy.

After you read the data in, you should check its integrity to avoid unexpected and artificial errors and delete missing values (null values).

In [2]:
## File paths

#data files
file_location = './data/experimental_data.xlsx'

#stop words
stopwords_location = './data/stopwords.txt'

In [3]:
## Check to make sure the dataset looks correct
try:
    data = pd.read_excel(file_location, encoding='latin1')
    print("{} Rows.  {} Columns.".format(*data.shape))
except:
    print("Dataset could not be loaded. Is the dataset missing?")

813 Rows.  2 Columns.


In [4]:
## Spot checks
indices = [0,333,777]

samples = pd.DataFrame(data.loc[indices, :], columns = data.keys()).reset_index(drop = True)
print("Sample Tickets:")
display(samples)

Sample Tickets:


,uid,GI_Author_merged
0,1.0,Get readers Expose sexual violence Report in a...
1,334.0,simply state that there was an altercation not...
2,778.0,just state what happened without bias


In [5]:
## Check number of null values in each column of the full dataset
pd.DataFrame(data.isnull().sum(), columns=['Number of NULL values'])

,Number of NULL values
uid,1
GI_Author_merged,0


In [6]:
## Remove missing (null) values from the data

#finding null values in the full dataset
print("=============Full Dataset=============")
data['GI_Author_merged'] = data['GI_Author_merged']

print('Number of rows in GI_Author_merged:', len(data['GI_Author_merged']))
print("-------------------")
print("Null Values in GI_Author_merged: {}".format(data['GI_Author_merged'].isnull().sum()))

#removing null values from the full dataset
GI_Author_merged = data['GI_Author_merged']

print("After removing Null Values in Full Dataset")
print("Null Values in GI_Author_merged: {}".format(data['GI_Author_merged'].isnull().sum()))

=============Full Dataset=============
Number of rows in GI_Author_merged: 813
-------------------
Null Values in GI_Author_merged: 0
After removing Null Values in Full Dataset
Null Values in GI_Author_merged: 0


### 2. Text preprocessing
This step is needed to generate a ‘bag-of-words’ LDA model and it includes: text tokenization and lemmatization, removal of Stop Words and words that appear in > 99% and < 1% of documents, including bigrams and trigrams, and generating Corpus and Dictionary.

**Tokenization** involves converting the text to lowercase, removing special characters, and punctuation from the text. Also, we should be careful to remove alphanumerics, numbers, words that appear in the corpus less than twice and extra spaces.


**Lemmatization** is used reduces the size of the vocabulary in the model. It transforms words to their lemma (e.g., assaulted -> assault). So that the model can analyze several inflected forms of a word as a single word. Also, lemmatization using Spacy allows to select certain part of speech words (e.g., noun, adj, vb, adv).


**Stop Word Removal** often is an important step to have a better input for modeling. Stop words are very common words in a language (e.g. a, an, the etc.). Note: you can edit the stop words txt file to add additional words to filter out.  We recommend filtering out as few stop words as possible, as even commonly occurring words can offer meaningful information, especially when responses are terse. However, depending on the specific characteristics of the textual data, stop word removal may be necessary to minimize model noise.

**Bigrams and Trigrams** are two and three consequent words that frequently co-occur together.

**Exclude terms in > 99% and < 1% of documents** is necessary to remove words that are contentless words in the documents. This allows to reduce model noise.


**Corpus** is our collection of documents (i.e., our textual questionnaire responses) and <br>
**Dictionary** is a list of unique words in the corpus. It takes each unique word in the corpus and assigns them an index.

In [7]:
## Convert the text to lowercase
GI_Author_merged = GI_Author_merged.str.lower()

print("=======Full Dataset==============\n")
print(GI_Author_merged.head(1))

=======Full Dataset==============

0    get readers expose sexual violence report in a...
Name: GI_Author_merged, dtype: object


In [8]:
## Remove special characters
GI_Author_merged_regex = [re.sub(r'\'', '', sent) for sent in GI_Author_merged]
GI_Author_merged_regex = [re.sub(r'_', ' ',  sent) for sent in GI_Author_merged_regex]
GI_Author_merged_regex = [re.sub(r'[^\w\s]', '', sent) for sent in GI_Author_merged_regex]

In [9]:
## Remove alphanumerics, numbers, words that appear in the corpus less than twice and extra spaces.
GI_Author_merged_regex = [re.sub(r'\d', '',  sent) for sent in GI_Author_merged_regex]
GI_Author_merged_regex = [re.sub(r'\W*\b\w{1,2}\b', '',  sent) for sent in GI_Author_merged_regex]
GI_Author_merged_regex = [re.sub(r'\S*@\S*\s?', '', sent) for sent in GI_Author_merged_regex]

print("=======Full Dataset==============\n")
print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_regex[:2])

=======Full Dataset==============


[INFO] GI_Author_merged....................

['get readers expose sexual violence report unbiased manner get interesting story pass class raise awareness report the facts get engagement write about something people care about expose the guy', '         ']


In [10]:
## Remove all punctuation and tokenize texts

#define helpful function
def tokenize(sentences):
    for sentence in sentences:
        yield(simple_preprocess(str(sentence), deacc=True))  # deacc=True removes all punctuation

#tolkenize full data set
GI_Author_merged_tokens = list(tokenize(GI_Author_merged_regex))


print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_tokens[:2])


[INFO] GI_Author_merged....................

[['get', 'readers', 'expose', 'sexual', 'violence', 'report', 'unbiased', 'manner', 'get', 'interesting', 'story', 'pass', 'class', 'raise', 'awareness', 'report', 'the', 'facts', 'get', 'engagement', 'write', 'about', 'something', 'people', 'care', 'about', 'expose', 'the', 'guy'], []]


In [11]:
## Lemmatize words, keeping only noun, adj, vb, adv

#define helpful function
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV', 'SCONJ', 'PRON', 'PART', 'NOUN', 'INTJ', 'AUX', 'ADV', 'ADP', 'ADJ']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc])
    return texts_out

#initialise Spacy
nlp = spacy.load('en', disable=['parser', 'ner'])

#lemmatize and select only noun, adj, vb, adv
GI_Author_merged_lemma = lemmatization(GI_Author_merged_tokens, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV', 'SCONJ', 'PRON', 'PART', 'NOUN', 'INTJ', 'AUX', 'ADV', 'ADP', 'ADJ'])
print(str(len(GI_Author_merged_lemma)))
print(GI_Author_merged_lemma[:4])

813
[['get', 'reader', 'expose', 'sexual', 'violence', 'report', 'unbiased', 'manner', 'get', 'interesting', 'story', 'pass', 'class', 'raise', 'awareness', 'report', 'the', 'fact', 'get', 'engagement', 'write', 'about', 'something', 'people', 'care', 'about', 'expose', 'the', 'guy'], [], [], []]


In [12]:
## Prepare to remove stop words
nltk.download('stopwords')
stopwords = set(nltk.corpus.stopwords.words('english'))
newStopWords =[str(x.strip()) for x in open(stopwords_location,'r').read().split('\n')]
stopwords.update(newStopWords)
print(len(stopwords))

219


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/hannahstevens/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
## Remove stop words 

#define helpful function
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stopwords] for doc in texts]

#remove stop words 
GI_Author_merged_stopwords = remove_stopwords(GI_Author_merged_lemma)

print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_stopwords[:2])


[INFO] GI_Author_merged....................

[['get', 'reader', 'expose', 'sexual', 'violence', 'report', 'unbiased', 'manner', 'get', 'interesting', 'story', 'pass', 'class', 'raise', 'awareness', 'report', 'fact', 'get', 'engagement', 'write', 'something', 'people', 'care', 'expose', 'guy'], []]


In [14]:
## Select Trigrams and Bigrams
            
GI_Author_merged_bigram = Phrases(GI_Author_merged_stopwords, min_count=3, delimiter=b' ', threshold=1)
GI_Author_merged_trigram = Phrases(GI_Author_merged_bigram[GI_Author_merged_stopwords], threshold=1)

GI_Author_merged_bigram_mod = gensim.models.phrases.Phraser(GI_Author_merged_bigram)
GI_Author_merged_trigram_mod = gensim.models.phrases.Phraser(GI_Author_merged_trigram)

for idx in range(len(GI_Author_merged_stopwords)):
    for token in GI_Author_merged_trigram_mod[GI_Author_merged_bigram_mod[GI_Author_merged_stopwords[idx]]]:
        #print(token)
        if ' ' in token:
            GI_Author_merged_stopwords[idx].append(token)
print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_stopwords[:2])


[INFO] GI_Author_merged....................

[['get', 'reader', 'expose', 'sexual', 'violence', 'report', 'unbiased', 'manner', 'get', 'interesting', 'story', 'pass', 'class', 'raise', 'awareness', 'report', 'fact', 'get', 'engagement', 'write', 'something', 'people', 'care', 'expose', 'guy', 'get reader', 'raise awareness', 'expose guy'], []]


In [15]:
## Generate Corpus
dictionary_GI_Author_merged = gensim.corpora.Dictionary(GI_Author_merged_stopwords)
dictionary_GI_Author_merged.filter_extremes(no_below=.01, no_above=0.99)

## Generate Dictionary
corpus_GI_Author_merged = [dictionary_GI_Author_merged.doc2bow(text) for text in GI_Author_merged_stopwords]

## Save Corpus and Dictionary on a local drive
pickle.dump(corpus_GI_Author_merged, open('./output/corpus_GI_Author_merged.pkl', 'wb'))
dictionary_GI_Author_merged.save('./output/dictionary_GI_Author_merged.gensim')

### 3. Model selection (Selecting the number of topics (k))
A Model is represented by model Hyperparameters that define prior distribution of the topics within each document, and prior distribution of the different words within each topic. These should be defined based on theoretical assumptions about how we think the topics are actually distributed amongst our data. LDA model from gensim library has the following Hyperparameters:
- **Beta** (referred to as 'eta' in gensim) = the [distribution of the] number of words per topic
- **Alpha** =  the [distribution of the] number of topics per document



Both alpha and eta can be set to ‘symmetric’, ‘asymmetric’, or ‘auto’, where:
- ‘auto’ = the model learns the best values for the hyperparameters as it is trained on more and more data (i.e., it learns an asymmetric prior from the corpus). See http://jonathan-huang.org/research/dirichlet/dirichlet.pdf for an overview             
- 'asymmetric' = uses a fixed, normalized asymmetric prior of 1.0 / k (number of topics)
- 'symmetric' = uses a distribution of 1 / k (number of topics)



In Bayesian statistics, we have to define the distributions (i.e., prior distributions) of unknown variables (e.g., ϕ and θ) before running the data analysis. These should be defined based on theoretical assumptions about how we think the topics are actually distributed amongst our data. In our case, it makes sense to assume that some documents discuss more/less topics than other documents; thus, we set the document-topic distribution to be asymmetric. 


We recommend setting alpha = 'auto' as it sets the distribution to be asymmetric, and learns the best alpha value (i.e., lowest perplexity scores) from the data itself. It also makes sense to assume that some topics contain more words than others. Thus, we recommend setting the distribution of the number of words per topic to be asymmetric as well.

In addition, gensim LDA model has the following parameters:
- **Passes** = number of laps the model goes through the entire corpus (Increasing the number of passes reduces model bias)
- **Chunksize** = number of documents to load into memory at a time (smaller chunk sizes save memory, but take longer to train)
- **Update_every** = number of chunks to process before maximizing your model 
- **Random state** = sets the seed to make the model reproducible
- **Number of topics (k)**

**Number of topics (k)** defines the LDA model. Researchers must tell the model how many (k) prominent goal inference topics to sort each ‘bag of words’ document into. Problematically, several different k-values might work. Thus, we use a metric called perplexity to help us to determine the optimal number of topics. The utility in perplexity comes from comparing perplexity values across models with differing k-values to pinpoint the best model (i.e., the model with the lowest perplexity score). 

We recommend testing the perplexity of the model with a variety of k values, and then running the final model using the k-value with the selected perplexity score. **Model perplexity** is a frequently used metric that gauges how well a model fits the data.

In [16]:
## Set model Hyper Parameters
k = [1,2,3,4,5,6,7,8,9,10,11,12]
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta= 'auto'
per_word_topics=True

lda_model_GI_Author_merged = []

In [17]:
## Get Perplexity Scores of Training Dataset
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

for i in k:
    lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                          id2word=dictionary_GI_Author_merged,
                                          num_topics=i, 
                                          random_state=random_state,
                                          update_every=update_every,
                                          chunksize=chunksize,
                                          passes=passes,
                                          iterations=iterations,
                                          alpha=alpha,
                                          eta=eta,                                                            
                                          per_word_topics=per_word_topics)

    print('\nPerplexity (num_topics = {}): '.format(i), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (num_topics = 1):  -6.19316089465453

Perplexity (num_topics = 2):  -6.253605099313884

Perplexity (num_topics = 3):  -6.294996413142641

Perplexity (num_topics = 4):  -6.332119621278861

Perplexity (num_topics = 5):  -6.3865077884785055

Perplexity (num_topics = 6):  -6.417232247865415

Perplexity (num_topics = 7):  -6.452010251100549

Perplexity (num_topics = 8):  -6.482557829453066

Perplexity (num_topics = 9):  -6.496667456006632

Perplexity (num_topics = 10):  -6.513583998167733

Perplexity (num_topics = 11):  -6.527812404505264

Perplexity (num_topics = 12):  -6.547015339646608


Usually, lower perplexity scores are indicative of increased model accuracy, and smaller k-values yield a more parsimonious set of topics. However, the perplexity score will often decrease as k increases. In these instances, it’s best to select the model that yields the highest perplexity value before the values flatten out. 

**Note: when selecting the optimal number of topics, we need to find a balance between overfitting and underfitting the model**

**Overfitting** (i.e., too many topics) can make it harder for human coders to label resulted topics since there is less coherence amongst the words in each topic. At the same time, resulting topics have less overlap in words.

**Underfitting** (i.e., too few topics) doesn't produce enough variance, limiting options for statistical analyses. Labeling resulting topics becomes easier since topics have more coherent list of words comprising each topic. At the same time, resulting topics have higher overlap in used words that leads to increased variance in the distribution of topics in each document.

**pyLDAvis visualization** of a model with selected value for k helps to asses Overfitting and Underfitting. Reading pyLDAvis:

Left pane:
- The area of each circle represents the prevalence of each topic over the entire corpus 
- The distance between the center of circles indicate the similarity between topics (i.e., inter-topic differences)

Right pane:
- If you hover over a particular topic on the left, the histogram on the right side lists the top 30 most relevant terms
- The widths of the gray bars represent the corpus-wide frequencies of each term, and the widths of the red bars represent the topic-specific frequencies of each term
- A slider at the top can adjust the relevance metric (λ); however, for our purposes, be sure it i set to λ = 1. For more information on the relevance metric, see library documentation. 

Documentation for this library can be found here: https://www.aclweb.org/anthology/W14-3110.pdf. 

**In the following steps we test LDA model hyper parameters for k in range [3-11].**

##### k = 3 topics

In [18]:
## Initializing LDA Models and Parameters
topic_number = 3
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")


lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 3):  -6.294996413142641


In [19]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 3...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 3

(0, '0.024*"male" + 0.023*"story" + 0.021*"show" + 0.018*"make" + 0.018*"give" + 0.017*"guy" + 0.016*"happen" + 0.015*"side" + 0.014*"accuse" + 0.012*"rape"')
(1, '0.034*"male" + 0.028*"sexual" + 0.026*"make" + 0.025*"assault" + 0.022*"show" + 0.016*"try" + 0.016*"reader" + 0.015*"people" + 0.015*"sexual assault" + 0.012*"want"')
(2, '0.028*"sexual" + 0.027*"assault" + 0.022*"try" + 0.020*"party" + 0.018*"public" + 0.018*"inform" + 0.016*"people" + 0.015*"guy" + 0.015*"sexual assault" + 0.014*"situation"')
GI_Author_Merged.....k = 3...................


/Users/hannahstevens/anaconda3/envs/first/lib/python3.6/site-packages/pyLDAvis/_prepare.py:257: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  return pd.concat([default_term_info] + list(topic_dfs))


##### k = 4 topics

In [20]:
## Initializing LDA Models and Parameters
topic_number = 4
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 4):  -6.332119621278861


In [21]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 4...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 4

(0, '0.038*"story" + 0.028*"give" + 0.024*"male" + 0.024*"side" + 0.019*"show" + 0.019*"happen" + 0.016*"tell" + 0.013*"accuse" + 0.011*"situation" + 0.011*"information"')
(1, '0.043*"make" + 0.035*"male" + 0.024*"show" + 0.021*"sexual" + 0.019*"reader" + 0.017*"assault" + 0.013*"accuse" + 0.012*"look" + 0.012*"try" + 0.012*"seem"')
(2, '0.044*"sexual" + 0.041*"assault" + 0.026*"inform" + 0.026*"sexual assault" + 0.024*"party" + 0.023*"public" + 0.018*"awareness" + 0.016*"bring" + 0.016*"try" + 0.015*"situation"')
(3, '0.030*"people" + 0.026*"guy" + 0.025*"try" + 0.025*"male" + 0.018*"show" + 0.017*"make" + 0.012*"happen" + 0.012*"want" + 0.012*"assault" + 0.011*"girl"')
GI_Author_Merged.....k = 4...................


##### k = 5 topics

In [22]:
## Initializing LDA Models and Parameters
topic_number = 5
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 5):  -6.3865077884785055


In [23]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 5...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 5

(0, '0.030*"male" + 0.028*"give" + 0.028*"show" + 0.028*"story" + 0.019*"side" + 0.019*"happen" + 0.015*"guy" + 0.014*"situation" + 0.012*"accuse" + 0.012*"report"')
(1, '0.043*"male" + 0.029*"make" + 0.026*"reader" + 0.026*"show" + 0.024*"sexual" + 0.022*"assault" + 0.012*"sexual assault" + 0.012*"incident" + 0.011*"look" + 0.010*"inform"')
(2, '0.051*"sexual" + 0.050*"assault" + 0.031*"sexual assault" + 0.031*"inform" + 0.027*"party" + 0.026*"public" + 0.024*"awareness" + 0.021*"bring" + 0.015*"warn" + 0.014*"try"')
(3, '0.032*"people" + 0.030*"try" + 0.025*"make" + 0.024*"guy" + 0.019*"male" + 0.016*"show" + 0.014*"happen" + 0.013*"sexual" + 0.012*"assault" + 0.011*"get"')
(4, '0.025*"want" + 0.021*"tell" + 0.020*"story" + 0.020*"make" + 0.017*"accuse" + 0.016*"ma

###### Note:  with k=5 topics appear to be relatively spread out, with no overlapping topics.  At the same time, the the list of words comprising each topic appears coherent enough to label. 

##### k = 6 topics

In [24]:
## Initializing LDA Models and Parameters
topic_number = 6
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 6):  -6.417232247865415


In [25]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 6...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 6

(0, '0.036*"give" + 0.035*"story" + 0.027*"male" + 0.022*"side" + 0.022*"show" + 0.020*"happen" + 0.014*"information" + 0.013*"situation" + 0.013*"guy" + 0.012*"perspective"')
(1, '0.046*"male" + 0.032*"reader" + 0.027*"show" + 0.026*"make" + 0.018*"sexual" + 0.017*"assault" + 0.013*"incident" + 0.011*"try" + 0.011*"side" + 0.010*"inform"')
(2, '0.058*"sexual" + 0.056*"assault" + 0.036*"sexual assault" + 0.029*"inform" + 0.025*"awareness" + 0.025*"public" + 0.024*"party" + 0.023*"bring" + 0.017*"warn" + 0.016*"try"')
(3, '0.033*"people" + 0.027*"make" + 0.026*"try" + 0.026*"male" + 0.022*"guy" + 0.019*"assault" + 0.016*"show" + 0.016*"sexual" + 0.015*"happen" + 0.013*"get"')
(4, '0.025*"make" + 0.025*"want" + 0.021*"accuse" + 0.019*"story" + 0.018*"tell" + 0.017*"mal

##### k = 7 topics

In [26]:
## Initializing LDA Models and Parameters
topic_number = 7
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 7):  -6.452010251100549


In [27]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 7...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 7

(0, '0.042*"male" + 0.032*"show" + 0.027*"give" + 0.024*"happen" + 0.017*"situation" + 0.016*"information" + 0.012*"accuse" + 0.011*"story" + 0.011*"background" + 0.009*"party"')
(1, '0.056*"make" + 0.051*"male" + 0.026*"reader" + 0.023*"show" + 0.021*"seem" + 0.019*"look" + 0.018*"like" + 0.016*"accuse" + 0.015*"bad" + 0.012*"victim"')
(2, '0.071*"sexual" + 0.068*"assault" + 0.044*"sexual assault" + 0.031*"inform" + 0.031*"party" + 0.030*"public" + 0.021*"warn" + 0.020*"awareness" + 0.017*"try" + 0.015*"situation"')
(3, '0.035*"people" + 0.026*"make" + 0.025*"male" + 0.023*"try" + 0.022*"guy" + 0.018*"show" + 0.014*"get" + 0.013*"victim" + 0.012*"happen" + 0.012*"assault"')
(4, '0.050*"want" + 0.028*"rape" + 0.016*"accuse" + 0.014*"male" + 0.014*"state" + 0.012*"try

##### k = 8 topics

In [28]:
## Initializing LDA Models and Parameters
topic_number = 8
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 8):  -6.482557829453066


In [29]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 8...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 8

(0, '0.051*"male" + 0.025*"show" + 0.020*"happen" + 0.011*"situation" + 0.011*"accuse" + 0.011*"paint" + 0.010*"perspective" + 0.010*"incident" + 0.009*"story" + 0.009*"report"')
(1, '0.041*"male" + 0.039*"sexual" + 0.036*"make" + 0.035*"assault" + 0.029*"show" + 0.025*"reader" + 0.019*"sexual assault" + 0.012*"incident" + 0.011*"look" + 0.011*"inform"')
(2, '0.042*"assault" + 0.041*"public" + 0.039*"sexual" + 0.039*"inform" + 0.035*"party" + 0.030*"sexual assault" + 0.023*"warn" + 0.022*"try" + 0.018*"inform public" + 0.018*"incident"')
(3, '0.042*"people" + 0.030*"try" + 0.026*"guy" + 0.023*"make" + 0.022*"show" + 0.019*"male" + 0.015*"assault" + 0.014*"sexual" + 0.014*"happen" + 0.012*"get"')
(4, '0.045*"want" + 0.023*"make" + 0.020*"accuse" + 0.018*"sexual" + 0.0

##### k = 9 topics

In [30]:
## Initializing LDA Models and Parameters
topic_number = 9
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 9):  -6.496667456006632


In [31]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 9...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 9

(0, '0.060*"male" + 0.013*"happen" + 0.011*"situation" + 0.011*"show" + 0.009*"put" + 0.009*"party" + 0.009*"girl" + 0.009*"could" + 0.009*"wrong" + 0.008*"take"')
(1, '0.071*"make" + 0.061*"male" + 0.028*"reader" + 0.023*"look" + 0.021*"seem" + 0.021*"like" + 0.021*"bad" + 0.017*"victim" + 0.014*"try" + 0.013*"guy"')
(2, '0.082*"sexual" + 0.078*"assault" + 0.052*"sexual assault" + 0.034*"inform" + 0.033*"party" + 0.031*"public" + 0.026*"awareness" + 0.023*"bring" + 0.018*"warn" + 0.015*"people"')
(3, '0.041*"people" + 0.028*"make" + 0.027*"try" + 0.021*"male" + 0.020*"guy" + 0.017*"get" + 0.015*"happen" + 0.014*"show" + 0.010*"think" + 0.010*"aware"')
(4, '0.051*"want" + 0.028*"rape" + 0.022*"people" + 0.017*"male" + 0.016*"get" + 0.014*"know" + 0.014*"accuse" + 0.0

##### k = 10 topics

In [32]:
## Initializing LDA Models and Parameters
topic_number =10
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 10):  -6.513583998167733


In [33]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 10...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 10

(0, '0.038*"male" + 0.018*"make" + 0.016*"accuse" + 0.012*"like" + 0.012*"happen" + 0.012*"story" + 0.011*"show" + 0.010*"put" + 0.010*"victim" + 0.009*"seem"')
(1, '0.054*"make" + 0.051*"male" + 0.037*"reader" + 0.018*"look" + 0.014*"incident" + 0.014*"seem" + 0.013*"bad" + 0.012*"like" + 0.011*"inform" + 0.011*"accuse"')
(2, '0.080*"sexual" + 0.077*"assault" + 0.054*"sexual assault" + 0.036*"inform" + 0.035*"public" + 0.032*"party" + 0.020*"bring" + 0.019*"awareness" + 0.017*"warn" + 0.016*"inform public"')
(3, '0.053*"people" + 0.032*"make" + 0.031*"try" + 0.026*"happen" + 0.020*"get" + 0.018*"aware" + 0.017*"know" + 0.015*"want" + 0.014*"male" + 0.012*"assault"')
(4, '0.041*"accuse" + 0.030*"rape" + 0.021*"make" + 0.019*"male" + 0.018*"bad" + 0.017*"light" + 0.0

##### k = 11 topics

In [34]:
## Initializing LDA Models and Parameters
topic_number =11
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 11):  -6.527812404505264


In [35]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 11...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] GI_Author_Merged Full Dataset Model Results....
***********************************************************************

[INFO] Num_topics: 11

(0, '0.053*"male" + 0.018*"situation" + 0.016*"make" + 0.016*"happen" + 0.016*"give" + 0.015*"show" + 0.013*"girl" + 0.013*"rape" + 0.012*"describe" + 0.012*"accuse"')
(1, '0.049*"male" + 0.048*"make" + 0.046*"reader" + 0.018*"blame" + 0.016*"put" + 0.014*"accuse" + 0.014*"victim" + 0.013*"look" + 0.012*"bad" + 0.012*"inform"')
(2, '0.061*"public" + 0.055*"inform" + 0.031*"inform public" + 0.027*"party" + 0.021*"situation" + 0.021*"bring" + 0.021*"try" + 0.019*"incident" + 0.018*"warn" + 0.015*"give"')
(3, '0.051*"people" + 0.026*"try" + 0.024*"make" + 0.023*"happen" + 0.021*"aware" + 0.014*"male" + 0.014*"show" + 0.011*"consent" + 0.010*"party" + 0.010*"may"')
(4, '0.026*"people" + 0.023*"want" + 0.020*"try" + 0.018*"rape" + 0.018*"get" + 0.016*"accuse" + 0.016*"pa

### 4. Saving selected model results
As k=5 topics yielded the lowest perplexity value before the trend flattened out at k=6 topics and with k=5 topics appear to be relatively spread out, with no overlapping topics, we determined a k=5-topic LDA model would produce the simplest model of the models that explain the data well


To save results from the LDA model with selected parameters and number of topics we 
- rerun the model with k=5
- generate a column that tells us which topic each response contributed the most to
- save the analysis results to an excel file for topic validation

In [36]:
## Initializing LDA Models and Parameters
topic_number = 5
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=800
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (topic_number = 5):  -6.384569057651238


In [37]:
## GI_Author_Merged Model Results
print("\n***********************************************************************")
print("[INFO] Goal Inferences Model Results....")
print("***********************************************************************")

print("\n[INFO] Number of topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("Goal Inferences .....k = 5...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)


***********************************************************************
[INFO] Goal Inferences Model Results....
***********************************************************************

[INFO] Number of topics: 5

(0, '0.030*"male" + 0.027*"give" + 0.027*"show" + 0.026*"story" + 0.019*"happen" + 0.018*"side" + 0.015*"guy" + 0.014*"situation" + 0.013*"accuse" + 0.012*"report"')
(1, '0.042*"male" + 0.030*"make" + 0.027*"reader" + 0.026*"sexual" + 0.025*"show" + 0.024*"assault" + 0.013*"sexual assault" + 0.012*"incident" + 0.011*"look" + 0.010*"try"')
(2, '0.051*"sexual" + 0.049*"assault" + 0.031*"inform" + 0.031*"sexual assault" + 0.027*"party" + 0.026*"public" + 0.024*"awareness" + 0.021*"bring" + 0.015*"try" + 0.015*"warn"')
(3, '0.032*"people" + 0.029*"try" + 0.025*"make" + 0.024*"guy" + 0.019*"male" + 0.017*"show" + 0.013*"happen" + 0.013*"sexual" + 0.012*"assault" + 0.011*"get"')
(4, '0.027*"want" + 0.021*"story" + 0.021*"tell" + 0.020*"make" + 0.018*"accuse" + 0.016*"male" + 0.015

In [41]:
## Generate a column that tells us which topic each response contributed the most to

#define helpful function
def format_topics_sentences(ldamodel, corpus, texts):
    # Init output
    sent_topics_df = pd.DataFrame()

    # Get the main topic of each document
    for i, row_list in enumerate(ldamodel[corpus]):
        row = row_list[0] if ldamodel.per_word_topics else row_list            
        row = sorted(row, key=lambda x: (x[1]), reverse=True)
        
        # Get the Dominant topic, Perc Contribution, and Keywords for each document
        raw_frame = {}
        for j, (topic_num, prop_topic) in enumerate(row):
            if j==0:
                raw_frame['Dominant'] = topic_num

            raw_frame['Topic' + str(topic_num)] = round(prop_topic, 4)
            
        df = pd.DataFrame(data=raw_frame, index=[0])
        sent_topics_df = sent_topics_df.append(df)

    return(sent_topics_df)


df_topic_sents_keywords_GI_Author_merged = format_topics_sentences(ldamodel=lda_model_GI_Author_merged, 
                                                                   corpus=corpus_GI_Author_merged, 
                                                                   texts=GI_Author_merged_stopwords)

#rename index of the dataframe
df_dominant_topic_GI_Author_merged = df_topic_sents_keywords_GI_Author_merged.reset_index()
df_dominant_topic_GI_Author_merged.index.name='Document_No';

print(df_dominant_topic_GI_Author_merged.head(812))

             index  Dominant  Topic0  Topic1  Topic2  Topic3  Topic4
Document_No                                                         
0                0         0  0.7117     NaN  0.2843     NaN     NaN
1                0         2  0.2182  0.1700  0.2428  0.1843  0.1847
2                0         2  0.2182  0.1700  0.2428  0.1843  0.1847
3                0         2  0.2182  0.1700  0.2428  0.1843  0.1847
4                0         2  0.2182  0.1700  0.2428  0.1843  0.1847
5                0         0  0.6263     NaN  0.3673     NaN     NaN
6                0         2     NaN     NaN  0.9860     NaN     NaN
7                0         2     NaN     NaN  0.9783     NaN     NaN
8                0         3     NaN     NaN     NaN  0.9889     NaN
9                0         1     NaN  0.9859     NaN     NaN     NaN
10               0         0  0.9867     NaN     NaN     NaN     NaN
11               0         0  0.9941     NaN     NaN     NaN     NaN
12               0         4  0.01

In [42]:
## Generate a data frame to export the results into
lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Dominant'])
topic0_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic0'])
topic1_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic1'])
topic2_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic2'])
topic3_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic3'])
topic4_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic4'])

GI_Author_merged = np.array(data['GI_Author_merged'])

uid = np.array(data['uid'])

results = { 
    'uid' : uid, 
    'GI_Author_merged': GI_Author_merged, 
    'lda_topics_GI_Author_merged': lda_topics_GI_Author_merged, 
    'topic0_contrib_lda_topics_GI_Author_merged':topic0_contrib_lda_topics_GI_Author_merged,
    'topic1_contrib_lda_topics_GI_Author_merged':topic1_contrib_lda_topics_GI_Author_merged,
    'topic2_contrib_lda_topics_GI_Author_merged':topic2_contrib_lda_topics_GI_Author_merged,
    'topic3_contrib_lda_topics_GI_Author_merged':topic3_contrib_lda_topics_GI_Author_merged,
    'topic4_contrib_lda_topics_GI_Author_merged':topic4_contrib_lda_topics_GI_Author_merged,

}

frame = pd.DataFrame(results, columns = [
                                        'uid',
                                        'GI_Author_merged', 'lda_topics_GI_Author_merged', 
                                        'topic0_contrib_lda_topics_GI_Author_merged',
                                        'topic1_contrib_lda_topics_GI_Author_merged',
                                        'topic2_contrib_lda_topics_GI_Author_merged',
                                        'topic3_contrib_lda_topics_GI_Author_merged',
                                        'topic4_contrib_lda_topics_GI_Author_merged',
                                        ])

In [43]:
## Export results to an .xlsx file
frame.to_excel("./output/lda_results_full_dataset_topic_num_5.xlsx")
frame.to_excel("./output/lda_results_All_data_topic_num_5.xlsx")